In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("Amazon_Review_Data.xlsx")

In [3]:
#EDA
df.head()

,Unique_ID,Category,Review_Header,Review_text,Rating,Own_Rating
0,136040,smartTv,Nice one,I liked it,5,Positive
1,134236,mobile,Huge battery life with amazing display,I bought the phone on Amazon and been using my...,5,Positive
2,113945,books,Four Stars,"Awesome book at reasonable price, must buy ......",4,Positive
3,168076,smartTv,Nice quality,good,5,Positive
4,157302,books,Nice book,"The book is fine,not bad,contains nice concept...",3,Neutral


In [4]:
df.shape

(60889, 6)

In [5]:
df.isnull().sum() # tells us how many missing values are there

Unique_ID         0
Category          0
Review_Header     6
Review_text      37
Rating            0
Own_Rating        0
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60889 entries, 0 to 60888
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Unique_ID      60889 non-null  int64 
 1   Category       60889 non-null  object
 2   Review_Header  60883 non-null  object
 3   Review_text    60852 non-null  object
 4   Rating         60889 non-null  int64 
 5   Own_Rating     60889 non-null  object
dtypes: int64(2), object(4)
memory usage: 2.8+ MB


In [7]:
df.describe()

,Unique_ID,Rating
count,60889.000000,60889.000000
mean,140444.000000,4.081148
std,17577.284607,1.342067
min,110000.000000,1.000000
25%,125222.000000,4.000000
50%,140444.000000,5.000000
75%,155666.000000,5.000000
max,170888.000000,5.000000


In [8]:
df["Rating"].value_counts().sort_index()

Rating
1     6979
2     2108
3     4366
4    12976
5    34460
Name: count, dtype: int64

In [9]:
pd.crosstab(df["Rating"],df["Own_Rating"])# this tells how Own_Rating was assigned from rating

Own_Rating,Negative,Neutral,Positive
Rating,,,
1,6979,0,0
2,2108,0,0
3,0,4366,0
4,0,0,12976
5,0,0,34460


In [10]:
df["Review_text"].sample(10,random_state=42)

9998                                              Fabulous
31306                                         Good product
29066    Sound is average. Camera is pathetic, battery ...
18603                                            Very good
43932    Awesome book for self improvement, after readi...
11578                                            very good
43686    The phone which we bought is not turning on an...
57037                                                 Nice
39394    Used product past 15days and battery back is g...
29874                         Battery life is ðŸ‘ðŸ‘ðŸ‘
Name: Review_text, dtype: object

In [11]:
df["Review_text"].duplicated().sum() # tell how may duplicate reviews are there in my dataset

np.int64(11007)

In [12]:
df["Review_text"].str.len().describe()# compute statistics the no. of character

count    60847.000000
mean       141.667034
std        311.677814
min          1.000000
25%         18.000000
50%         61.000000
75%        158.000000
max      18294.000000
Name: Review_text, dtype: float64

In [13]:
df["Own_Rating"].value_counts()# how many negative pso nutra reviews we have 

Own_Rating
Positive    47436
Negative     9087
Neutral      4366
Name: count, dtype: int64

In [14]:
df["Category"].value_counts()

Category
mobile                22749
mobile accessories    14811
smartTv               14624
refrigerator           4791
books                  3914
Name: count, dtype: int64

In [15]:
df["Review_text"].isnull().sum()

np.int64(37)

In [16]:
df_clean = df.copy()

In [17]:
# df_clean[df_clean["Review_text"].isnull()]# those are that 37 vlues missing values

In [18]:
df_clean = df_clean.dropna(subset=["Review_text"]) 

In [19]:
# df_clean["Review_text"].head(20).tolist()

In [20]:
df_clean["Review_text"].map(type).value_counts()

Review_text
<class 'str'>                  60847
<class 'datetime.datetime'>        3
<class 'int'>                      2
Name: count, dtype: int64

In [21]:
df_clean[df_clean["Review_text"].map(type) != str][["Review_text","Rating","Own_Rating"]] # show us the rows where Review_text is not a string

,Review_text,Rating,Own_Rating
10300,2026-05-10 00:00:00,1,Negative
13136,2026-10-10 00:00:00,5,Positive
29643,10,5,Positive
39108,679794,5,Positive
50444,2026-10-10 00:00:00,5,Positive


In [22]:
df_clean = df_clean[df_clean["Review_text"].map(type) == str] # this basically returns only string vlaues in review_text

In [23]:
df_clean["Review_text"].map(type).value_counts()

Review_text
<class 'str'>    60847
Name: count, dtype: int64

In [24]:
df_clean["Review_text"] = df_clean["Review_text"].str.lower() # it converts all the lines in lowercase

In [25]:
import re
df_clean["Review_text"] = df_clean["Review_text"].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]',' ',x))# purpose remove pantuation, symbols and 
# emogis While kipping letters,no. and spaces.

In [26]:
df_clean["Review_text"] = df_clean["Review_text"].str.replace(r'\s+',' ', regex=True).str.strip() 
# normalize multiple spaces and remove unnecessory spaces at the begginning/s=end.

In [27]:
df_clean["Review_text"].head(20)

0                                            i liked it
1     i bought the phone on amazon and been using my...
2             awesome book at reasonable price must buy
3                                                  good
4     the book is fine not bad contains nice concept...
5     nice tv and pic quality good custmer srrvice m...
6     the iphone 7 is legitimately among the most in...
7            20000 mah what more you need super product
8     the company should give more bettany backup an...
9                                       very good phone
10                                          good option
11    redmi note 6 pro is the best mobile at the bes...
12                                                 good
13    product is good as expected but after sale ser...
14                                         good product
15                                           good phone
16    i m a fan of alexa fire stick its a value for ...
17                      delivered in time worth 

In [28]:
#Tokenisation
df_clean["tokens"] = df_clean["Review_text"].str.split() 

In [29]:
stop_words = {"is","the","a","an","and","of","to","in","it"}
df_clean["tokens"] = df_clean["tokens"].apply(lambda tokens: [word for word in tokens if word not in stop_words])

In [30]:
# now we do lemmatization  like playing --> play
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [31]:
df_clean["lemmas"] = df_clean["tokens"].apply(lambda tokens:[lemmatizer.lemmatize(word) for word in tokens]) 

In [32]:
df_clean[["Review_text","tokens","lemmas"]].head()

,Review_text,tokens,lemmas
0,i liked it,"[i, liked]","[i, liked]"
1,i bought the phone on amazon and been using my...,"[i, bought, phone, on, amazon, been, using, my...","[i, bought, phone, on, amazon, been, using, my..."
2,awesome book at reasonable price must buy,"[awesome, book, at, reasonable, price, must, buy]","[awesome, book, at, reasonable, price, must, buy]"
3,good,[good],[good]
4,the book is fine not bad contains nice concept...,"[book, fine, not, bad, contains, nice, concept...","[book, fine, not, bad, contains, nice, concept..."


In [33]:
# we will use TF-IDF bcz it want str
df_clean["text_for_tfidf"] = df_clean["lemmas"].apply(" ".join) # this gives lemmatizer text into normal string

In [34]:
df_clean[["lemmas", "text_for_tfidf"]].head()

,lemmas,text_for_tfidf
0,"[i, liked]",i liked
1,"[i, bought, phone, on, amazon, been, using, my...",i bought phone on amazon been using my samsung...
2,"[awesome, book, at, reasonable, price, must, buy]",awesome book at reasonable price must buy
3,[good],good
4,"[book, fine, not, bad, contains, nice, concept...",book fine not bad contains nice concept nicely...


In [35]:
df_clean = df_clean.drop_duplicates(
    subset="Review_text",
    keep="first"
).reset_index(drop=True)

print("Rows after removing duplicates:", len(df_clean))
print("Duplicate reviews remaining:", df_clean["Review_text"].duplicated().sum())

Rows after removing duplicates: 48264
Duplicate reviews remaining: 0


In [36]:

from sklearn.model_selection import train_test_split

X_temp_text, X_unseen_text, Y_temp, Y_unseen = train_test_split(
    df_clean["text_for_tfidf"],
    df_clean["Own_Rating"],
    test_size=0.005,
    random_state=42,
    stratify=df_clean["Own_Rating"]
)

X_train_text, X_val_text, Y_train, Y_val = train_test_split(
    X_temp_text,
    Y_temp,
    test_size=0.19598,
    random_state=42,
    stratify=Y_temp
)

print("Training:", len(X_train_text))
print("Validation:", len(X_val_text))
print("Unseen:", len(X_unseen_text))

Training: 38610
Validation: 9412
Unseen: 242


In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train = tfidf.fit_transform(X_train_text)
X_val = tfidf.transform(X_val_text)
X_unseen = tfidf.transform(X_unseen_text)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_unseen:", X_unseen.shape)

X_train: (38610, 21891)
X_val: (9412, 21891)
X_unseen: (242, 21891)


In [38]:
from sklearn.linear_model import LogisticRegression                # -----------------LOGISTIC REGRESSION--------------------

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, Y_train)

Y_val_pred_lr = lr_model.predict(X_val)

In [39]:
from sklearn.metrics import accuracy_score, classification_report

print("Logistic Regression")
print("Accuracy:", accuracy_score(Y_val, Y_val_pred_lr))

print("\nClassification Report:")
print(classification_report(Y_val, Y_val_pred_lr))

Logistic Regression
Accuracy: 0.8535911602209945

Classification Report:
              precision    recall  f1-score   support

    Negative       0.75      0.75      0.75      1677
     Neutral       0.33      0.04      0.08       736
    Positive       0.88      0.96      0.92      6999

    accuracy                           0.85      9412
   macro avg       0.65      0.59      0.58      9412
weighted avg       0.82      0.85      0.82      9412



In [40]:
from sklearn.linear_model import LogisticRegression

balanced_lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

balanced_lr.fit(X_train, Y_train)

Y_val_pred_balanced = balanced_lr.predict(X_val)

In [41]:
from sklearn.metrics import accuracy_score, classification_report           # -----------------BALANCED LOGISTIC REGRESSION--------------------

print("Balanced Logistic Regression")
print("Accuracy:", accuracy_score(Y_val, Y_val_pred_balanced))

print("\nClassification Report:")
print(classification_report(Y_val, Y_val_pred_balanced))

Balanced Logistic Regression
Accuracy: 0.7724181895452613

Classification Report:
              precision    recall  f1-score   support

    Negative       0.68      0.76      0.72      1677
     Neutral       0.20      0.43      0.27       736
    Positive       0.95      0.81      0.88      6999

    accuracy                           0.77      9412
   macro avg       0.61      0.67      0.62      9412
weighted avg       0.84      0.77      0.80      9412



In [42]:
import pandas as pd

model_results = pd.read_excel("traditional_ml_results.xlsx")

model_results

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Logistic Regression,0.853591,0.653693,0.585731,0.583032,0.824930
1,Balanced Logistic Regression,0.772418,0.611529,0.667115,0.623022,0.800603
2,Multinomial Naive Bayes,0.790374,0.549849,0.427139,0.436336,0.729523
3,Linear SVM,0.848279,0.624174,0.581784,0.575685,0.820720


In [43]:
print("Logistic Regression: " ,"logistic_model" in globals())
print("Balanced Logistic Regression: " , "balanced_logistic_model" in globals())

Logistic Regression:  False
Balanced Logistic Regression:  False


In [44]:
import joblib

joblib.dump(lr_model,"logistic_regression_model.pkl")
joblib.dump(balanced_lr,"balanced_logistic_regression_model.pkl")

print("both Logistic Regression models saved successfully!")

both Logistic Regression models saved successfully!


In [45]:
#DEMO SAMPLE 

review = "The product quality is excellent and I am very happy with my purchase."

# Convert review into TF-IDF
review_tfidf = tfidf.transform([review])

# Class probabilities
lr_probabilities = lr_model.predict_proba(review_tfidf)[0]
balanced_probabilities = balanced_lr.predict_proba(review_tfidf)[0]

# Predicted class
lr_prediction = lr_model.classes_[lr_probabilities.argmax()]
balanced_prediction = balanced_lr.classes_[balanced_probabilities.argmax()]

# Confidence
lr_confidence = lr_probabilities.max()
balanced_confidence = balanced_probabilities.max()

print("Review:", review)

print("\nLogistic Regression")
print("Sentiment:", lr_prediction)
print(f"Confidence: {lr_confidence * 100:.2f}%")

print("\nBalanced Logistic Regression")
print("Sentiment:", balanced_prediction)
print(f"Confidence: {balanced_confidence * 100:.2f}%")

Review: The product quality is excellent and I am very happy with my purchase.

Logistic Regression
Sentiment: Positive
Confidence: 99.64%

Balanced Logistic Regression
Sentiment: Positive
Confidence: 99.06%
